### Neuronska mreža

In [ ]:
import pandas as pd

df=pd.read_csv("../data/processed/final.csv")
df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,count_submitted_TMA,std_score_CMA,std_score_TMA,total_assessments_CMA,total_assessments_TMA,submission_rate_TMA,submission_rate_CMA,total_clicks,last_active_day,unique_activities
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,2.0,0.0,4.949747,0.0,2.0,1.0,0.0,587.0,87.0,6.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,2.0,0.0,1.414214,0.0,2.0,1.0,0.0,821.0,90.0,6.0
2,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,2.0,0.0,0.707107,0.0,2.0,1.0,0.0,992.0,90.0,7.0
3,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,2.0,0.0,27.577164,0.0,2.0,1.0,0.0,684.0,90.0,7.0
4,AAA,2013J,38053,M,Wales,A Level or Equivalent,80-90%,35-55,0,60,...,2.0,0.0,7.071068,0.0,2.0,1.0,0.0,1060.0,86.0,7.0


In [18]:
df['target']=df['final_result'].isin(["Fail", "Withdrawn"]).astype(int)
df['year']=df['code_presentation'].astype(str).str[:4]

train_df=df[df['year']=="2013"].copy()
test_df=df[df["year"]=="2014"].copy()

X_train=train_df.drop(columns=["final_result", "target", "id_student", "year"],errors='ignore')
y_train=train_df['target']
X_test=test_df.drop(columns=["final_result", "target", "id_student", "year"],errors='ignore')
y_test=test_df['target']

In [19]:
import joblib

model = joblib.load("../models/final_model.pkl")
preprocessor = model.named_steps["preprocessor"]

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(type(X_train_processed))
print(type(X_test_processed))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [20]:
import torch

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [21]:
print(X_train_tensor.shape)
print(y_train_tensor.shape)
print(X_test_tensor.shape)
print(y_test_tensor.shape)

torch.Size([11055, 61])
torch.Size([11055, 1])
torch.Size([14507, 61])
torch.Size([14507, 1])


In [22]:
import torch.nn as nn

class StudentRiskNN(nn.Module):
    def __init__(self, input_dim):
        super(StudentRiskNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.network(x)

In [23]:
import random
import numpy as np
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [24]:
input_dim = X_train_tensor.shape[1]
model_nn = StudentRiskNN(input_dim)
model_nn

StudentRiskNN(
  (network): Sequential(
    (0): Linear(in_features=61, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [25]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_nn.parameters(), lr=0.001)

In [26]:

print("Broj parametara:", sum(p.numel() for p in model_nn.parameters()))

Broj parametara: 6081


In [27]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [28]:
epochs = 50
train_losses = []

for epoch in range(epochs):
    model_nn.train()
    epoch_loss = 0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model_nn(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 10/50, Loss: 0.4249
Epoch 20/50, Loss: 0.4057
Epoch 30/50, Loss: 0.3956
Epoch 40/50, Loss: 0.3758
Epoch 50/50, Loss: 0.3684


In [29]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

threshold=joblib.load("../models/threshold.pkl")
model_nn.eval()

with torch.no_grad():
    y_logits = model_nn(X_test_tensor)
    y_proba_nn = torch.sigmoid(y_logits).numpy().flatten()
    y_pred_nn = (y_proba_nn >= threshold).astype(int)

print(f"Accuracy: {accuracy_score(y_test, y_pred_nn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nn, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_nn, zero_division=0):.4f}")
print(f"F1: {f1_score(y_test, y_pred_nn):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba_nn):.4f}")

Accuracy: 0.7396
Precision: 0.6536
Recall: 0.7156
F1: 0.6832
ROC AUC: 0.8145


In [30]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred_nn)

array([[6656, 2159],
       [1619, 4073]])